In [3]:
import pandas as pd
import numpy as np

In [2]:
df = pd.DataFrame({
    "Size": [500, 600, 800, 1000, 1200, 1500, 1800, 2000],
    "Price": [20, 25, 30, 35, 45, 50, 60, 65]
})

print(df)

   Size  Price
0   500     20
1   600     25
2   800     30
3  1000     35
4  1200     45
5  1500     50
6  1800     60
7  2000     65


In [4]:
ymean = np.mean(df['Price'])

In [34]:
df['Gradient'] = ymean-df['Price']

In [36]:
df['Hessain'] = 1

In [37]:
df

,Size,Price,Bin,Price_squared,Gradient,Hessain
0,500,20,"(498.5, 875.0]",400,21.25,1
1,600,25,"(498.5, 875.0]",625,16.25,1
2,800,30,"(498.5, 875.0]",900,11.25,1
3,1000,35,"(875.0, 1250.0]",1225,6.25,1
4,1200,45,"(875.0, 1250.0]",2025,-3.75,1
5,1500,50,"(1250.0, 1625.0]",2500,-8.75,1
6,1800,60,"(1625.0, 2000.0]",3600,-18.75,1
7,2000,65,"(1625.0, 2000.0]",4225,-23.75,1


In [13]:
loss_root =sum((df['Price']-ymean)**2)

In [38]:
bins = pd.cut(df['Size'],bins=4)

In [39]:
bins

0      (498.5, 875.0]
1      (498.5, 875.0]
2      (498.5, 875.0]
3     (875.0, 1250.0]
4     (875.0, 1250.0]
5    (1250.0, 1625.0]
6    (1625.0, 2000.0]
7    (1625.0, 2000.0]
Name: Size, dtype: category
Categories (4, interval[float64, right]): [(498.5, 875.0] < (875.0, 1250.0] < (1250.0, 1625.0] < (1625.0, 2000.0]]

In [40]:

df['Bin'] = bins

In [41]:
df

,Size,Price,Bin,Price_squared,Gradient,Hessain
0,500,20,"(498.5, 875.0]",400,21.25,1
1,600,25,"(498.5, 875.0]",625,16.25,1
2,800,30,"(498.5, 875.0]",900,11.25,1
3,1000,35,"(875.0, 1250.0]",1225,6.25,1
4,1200,45,"(875.0, 1250.0]",2025,-3.75,1
5,1500,50,"(1250.0, 1625.0]",2500,-8.75,1
6,1800,60,"(1625.0, 2000.0]",3600,-18.75,1
7,2000,65,"(1625.0, 2000.0]",4225,-23.75,1


In [19]:
df.groupby('Bin',observed=True)

In [42]:
df['Price_squared'] = df['Price'] ** 2

stats = df.groupby('Bin', observed=True).agg(
    Count=('Price', 'count'),
    Price_sum=('Price', 'sum'),
    Price_squared_sum=('Price_squared', 'sum'),
    Gradient_sum = ('Gradient','sum'),
    Hessian_sum = ('Hessain','sum')
)

print(stats)

                  Count  Price_sum  Price_squared_sum  Gradient_sum  \
Bin                                                                   
(498.5, 875.0]        3         75               1925         48.75   
(875.0, 1250.0]       2         80               3250          2.50   
(1250.0, 1625.0]      1         50               2500         -8.75   
(1625.0, 2000.0]      2        125               7825        -42.50   

                  Hessian_sum  
Bin                            
(498.5, 875.0]              3  
(875.0, 1250.0]             2  
(1250.0, 1625.0]            1  
(1625.0, 2000.0]            2  


In [43]:
stats['Cumulative_count'] = stats['Count'].cumsum()

In [44]:
stats

,Count,Price_sum,Price_squared_sum,Gradient_sum,Hessian_sum,Cumulative_count
Bin,,,,,,
"(498.5, 875.0]",3,75,1925,48.75,3,3
"(875.0, 1250.0]",2,80,3250,2.50,2,5
"(1250.0, 1625.0]",1,50,2500,-8.75,1,6
"(1625.0, 2000.0]",2,125,7825,-42.50,2,8


In [45]:
stats['Cumulative_price_sum'] = stats['Price_sum'].cumsum()

In [46]:
stats['Cumulative_squared_price_sum'] = stats['Price_squared_sum'].cumsum()

In [47]:
stats['Cumulative_gradient_sum'] = stats['Gradient_sum'].cumsum()

In [48]:
stats['Cumulative_hessain_sum'] = stats['Hessian_sum'].cumsum()

In [49]:
stats

,Count,Price_sum,Price_squared_sum,Gradient_sum,Hessian_sum,Cumulative_count,Cumulative_price_sum,Cumulative_squared_price_sum,Cumulative_gradient_sum,Cumulative_hessain_sum
Bin,,,,,,,,,,
"(498.5, 875.0]",3,75,1925,48.75,3,3,75,1925,48.75,3
"(875.0, 1250.0]",2,80,3250,2.50,2,5,155,5175,51.25,5
"(1250.0, 1625.0]",1,50,2500,-8.75,1,6,205,7675,42.50,6
"(1625.0, 2000.0]",2,125,7825,-42.50,2,8,330,15500,0.00,8


In [26]:
len(stats)

4

[np.float64(620.0),
 np.float64(486.66666666666606),
 np.float64(683.333333333333)]

In [32]:
g = df['Price']-ymean

In [33]:
g

0   -21.25
1   -16.25
2   -11.25
3    -6.25
4     3.75
5     8.75
6    18.75
7    23.75
Name: Price, dtype: float64

In [54]:
def gain(g,h):
    return (g**2/(h+1))

In [59]:
last_grad  = stats['Cumulative_gradient_sum'].iloc[-1]
last_hessain = stats['Cumulative_hessain_sum'].iloc[-1]

gains = []
for i in range(0,len(stats)-1):
    lgrad = stats['Cumulative_gradient_sum'].iloc[i]
    rgrad = last_grad-lgrad
    lhess = stats['Cumulative_hessain_sum'].iloc[i]
    rhess = last_hessain-lhess
    g = 0.5*(gain(lgrad,lhess)+gain(rgrad,rhess))
    gains.append(g)

In [64]:
best_split = np.argmax(gains)

In [65]:
best_split

np.int64(1)

In [74]:
y_new = []
last_count = stats['Cumulative_count'].iloc[-1]
last_sum = stats['Cumulative_price_sum'].iloc[-1]
last_grad  = stats['Cumulative_gradient_sum'].iloc[-1]
last_hessain = stats['Cumulative_hessain_sum'].iloc[-1]
lc = stats['Cumulative_count'].iloc[best_split]
rc = last_count-lc
lsum = stats['Cumulative_price_sum'].iloc[best_split]
rsum = last_sum-lsum
lgrad = stats.Cumulative_gradient_sum.iloc[best_split]
rgrad = last_grad-lgrad
lhess = stats.Cumulative_hessain_sum.iloc[best_split]
rhess = last_hessain-lhess
lefty = ymean-0.1*(lgrad/(lhess+1))
righty = ymean-0.1*(rgrad/(rhess+1))
y_new.append(lefty)
y_new.append(righty)



In [75]:
y_new

[np.float64(40.395833333333336), np.float64(42.53125)]

In [78]:
df['Prediction'] = np.where(
    df['Size'] <= 1250,
    y_new[0],
    y_new[1]
)

In [79]:
df

,Size,Price,Bin,Price_squared,Gradient,Hessain,Prediction
0,500,20,"(498.5, 875.0]",400,21.25,1,40.395833
1,600,25,"(498.5, 875.0]",625,16.25,1,40.395833
2,800,30,"(498.5, 875.0]",900,11.25,1,40.395833
3,1000,35,"(875.0, 1250.0]",1225,6.25,1,40.395833
4,1200,45,"(875.0, 1250.0]",2025,-3.75,1,40.395833
5,1500,50,"(1250.0, 1625.0]",2500,-8.75,1,42.531250
6,1800,60,"(1625.0, 2000.0]",3600,-18.75,1,42.531250
7,2000,65,"(1625.0, 2000.0]",4225,-23.75,1,42.531250


In [80]:
import pandas as pd
import numpy as np

def gain(g, h, lambda_reg=1):
    return g**2 / (h + lambda_reg)


def train_xgboost(df, n_trees, learning_rate=0.1, lambda_reg=1, bins=4):
    df = df.copy()

    ymean = df['Price'].mean()
    df['Prediction'] = ymean
    df['Bin'] = pd.cut(df['Size'], bins=bins)
    df['Hessain'] = 1

    trees = []

    for tree in range(n_trees):
        df['Gradient'] = df['Prediction'] - df['Price']

        stats = df.groupby('Bin', observed=True).agg(
            Count=('Price', 'count'),
            Gradient_sum=('Gradient', 'sum'),
            Hessian_sum=('Hessain', 'sum')
        )

        stats['Cumulative_count'] = stats['Count'].cumsum()
        stats['Cumulative_gradient_sum'] = stats['Gradient_sum'].cumsum()
        stats['Cumulative_hessian_sum'] = stats['Hessian_sum'].cumsum()

        last_grad = stats['Cumulative_gradient_sum'].iloc[-1]
        last_hessian = stats['Cumulative_hessian_sum'].iloc[-1]

        gains = []

        for i in range(len(stats) - 1):
            lgrad = stats['Cumulative_gradient_sum'].iloc[i]
            rgrad = last_grad - lgrad

            lhess = stats['Cumulative_hessian_sum'].iloc[i]
            rhess = last_hessian - lhess

            parent_score = gain(last_grad, last_hessian, lambda_reg)
            left_score = gain(lgrad, lhess, lambda_reg)
            right_score = gain(rgrad, rhess, lambda_reg)

            split_gain = 0.5 * (
                left_score + right_score - parent_score
            )

            gains.append(split_gain)

        best_split = np.argmax(gains)
        threshold = stats.index[best_split].right

        lgrad = stats['Cumulative_gradient_sum'].iloc[best_split]
        lhess = stats['Cumulative_hessian_sum'].iloc[best_split]

        rgrad = last_grad - lgrad
        rhess = last_hessian - lhess

        left_weight = -lgrad / (lhess + lambda_reg)
        right_weight = -rgrad / (rhess + lambda_reg)

        df['Prediction'] += np.where(
            df['Size'] <= threshold,
            learning_rate * left_weight,
            learning_rate * right_weight
        )

        trees.append({
            'threshold': threshold,
            'left_weight': left_weight,
            'right_weight': right_weight,
            'gain': gains[best_split]
        })

    return df, trees

In [85]:
result, trees = train_xgboost(df, n_trees=300)

print(result[['Size', 'Price', 'Prediction']])


   Size  Price  Prediction
0   500     20   25.000001
1   600     25   25.000001
2   800     30   25.000001
3  1000     35   40.000000
4  1200     45   40.000000
5  1500     50   50.000000
6  1800     60   62.499999
7  2000     65   62.499999


In [86]:
mse = np.mean((result['Price'] - result['Prediction']) ** 2)
print("Training MSE:", mse)

Training MSE: 14.06250000000065
